# Day 3 · Session 3 — Train Your First Model on Azure

Run each cell with **Shift + Enter**, from top to bottom.

Before you start, make sure:
- your compute instance is **Running** (green dot)
- the kernel at the top-right says **Python 3.10 - SDK v2** (or similar)
- `soldiers_fitness.csv` is uploaded next to this notebook

## Cell 1 — Check where we are running

This notebook is running on the compute instance you started — a computer in a
Microsoft data centre, not your laptop.

In [ ]:
import platform, multiprocessing

print("Python :", platform.python_version())
print("System :", platform.system())
print("CPUs   :", multiprocessing.cpu_count())
print()
print("This is running on your Azure compute instance.")

## Cell 2 — Load the data

We read the CSV that sits next to this notebook. (Later you will read it straight from
Blob Storage instead — see Cell 9.)

In [ ]:
import pandas as pd

df = pd.read_csv("soldiers_fitness.csv")

print(f"{len(df)} rows, {len(df.columns)} columns")
df.head()

## Cell 3 — Look at it before you model it

Always look at your data first. A model can only learn what is actually there.

In [ ]:
print("How many people at each fitness level?")
print(df["level"].value_counts())
print()
print("Averages by level:")
print(df.groupby("level")[["age", "pushups", "run_5km_min"]].mean().round(1))

Notice the pattern already visible: people rated `high` do more pushups and run
faster. That is exactly the pattern the model will learn.

## Cell 4 — Split into questions and answer

- **X** = the information we give the model (the questions)
- **y** = what we want it to predict (the answer)

In [ ]:
X = df[["age", "pushups", "run_5km_min"]]    # the questions
y = df["level"]                              # the answer

print("X (questions):")
print(X.head(3))
print()
print("y (answer):")
print(y.head(3).tolist())

## Cell 5 — Split into training and testing

We hold back 30% of the rows. The model never sees them during training, so testing on
them tells us whether it really learned — or just memorised.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Training on {len(X_train)} rows")
print(f"Testing on  {len(X_test)} rows")

## Cell 6 — Train the model

**This is training.** Two lines. On this tiny dataset it takes a fraction of a second.

On Day 1 you calculated that training a model like PaLM takes weeks on TPU pods.
The difference is the amount of data, not the idea.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)          # <-- this is training

print("Training finished.")

## Cell 7 — How good is it?

In [ ]:
accuracy = model.score(X_test, y_test)

print(f"Accuracy on unseen data: {accuracy:.0%}")

You will probably see 100%.

**Do not be impressed.** With only 20 rows and an obvious pattern, this is easy. Real
datasets have thousands of rows, noise, and contradictions, and 70–85% is often a good
result. A perfect score on a tiny dataset usually means the problem was too easy — or
that something is wrong.

This is an important habit: always ask *why* a number is good, not just whether it is.

## Cell 8 — Inference: ask it about a new person

**This is inference** — using the trained model to get an answer about someone it has
never seen.

In [ ]:
new_person = pd.DataFrame({
    "age": [25],
    "pushups": [48],
    "run_5km_min": [23],
})

prediction = model.predict(new_person)

print("A 25-year-old doing 48 pushups and running 5km in 23 minutes")
print("is predicted to be:", prediction[0])

### See the rules it learned

A decision tree is one of the few models you can read directly.

In [ ]:
from sklearn.tree import export_text

print(export_text(model, feature_names=list(X.columns)))

Read it as a set of if/else questions. The model worked these thresholds out by
itself — nobody wrote them.

## Cell 9 — (Optional) read the data from Blob Storage instead

This is how it works in a real project: the data lives in storage, not next to the
notebook. Change the storage account name to yours.

In [ ]:
# Uncomment and edit to try it
#
# import io
# from azure.identity import DefaultAzureCredential
# from azure.storage.blob import BlobServiceClient
#
# STORAGE_ACCOUNT = "stabebe2026cli"     # <-- your storage account
#
# service = BlobServiceClient(
#     account_url=f"https://{STORAGE_ACCOUNT}.blob.core.windows.net",
#     credential=DefaultAzureCredential())
#
# raw = service.get_blob_client("data", "soldiers_fitness.csv").download_blob().readall()
# df_cloud = pd.read_csv(io.BytesIO(raw))
# print(f"Loaded {len(df_cloud)} rows straight from Blob Storage")
# df_cloud.head()

## Cell 10 — Save the model to a file

A model inside a notebook disappears when the notebook closes. First save it to a file.

In [ ]:
import joblib

joblib.dump(model, "fitness_model.pkl")

print("Saved as fitness_model.pkl")

## Cell 11 — Register the model in your workspace

Registering gives the model a **name** and a **version** and stores it in the workspace,
so it survives, can be shared, and can later be deployed.

In [ ]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential

# On an Azure ML compute instance this finds your workspace automatically
ml_client = MLClient.from_config(credential=DefaultAzureCredential())

registered = ml_client.models.create_or_update(
    Model(
        path="fitness_model.pkl",
        name="fitness-model",
        type=AssetTypes.CUSTOM_MODEL,
        description="Predicts fitness level from age, pushups and 5km time",
    )
)

print(f"Registered '{registered.name}' version {registered.version}")

**Check it worked:** in ML Studio, click **Models** in the left menu. Your model is
listed with version 1.

If Cell 11 fails, it is almost always because the notebook is not running on the compute
instance inside the workspace. That is fine — the important learning was Cells 6 and 8.

---

# NOW STOP YOUR COMPUTE INSTANCE

This is not optional. A running compute instance bills every minute, including overnight.

1. In ML Studio, click **Compute** in the left menu
2. Tick the box next to your instance
3. Click **Stop**
4. Wait until it says **Stopped** — not *Stopping*

Your notebooks and files stay exactly where they are. A stopped instance costs nothing.

---

## What you did today

| Step | What it was |
|---|---|
| `model.fit()` | **Training** — the model learned the pattern |
| `model.score()` | **Evaluation** — how well it did on unseen data |
| `model.predict()` | **Inference** — using it to answer a new question |
| `models.create_or_update()` | **Registration** — saving it properly |

**Next:** Session 4 — using a model Microsoft already trained, with no compute instance
at all.